# LightGBM Regressor from Scratch

The math behind LightGBM is identical to XGBoost — same gradients, same hessians, same gain formula, same leaf output. What changes is one thing: **how the tree decides where to grow next**.

This notebook focuses entirely on that difference and what it means in practice.

In [6]:
from wrapped_models.lightgbm_regressor import LightGBMRegressor
from wrapped_models.xgboost_regressor import XGBoostRegressor
from wrapped_models.lightgbm_tree import Node
import numpy as np
import pandas as pd

## Level-wise vs Leaf-wise Growth

XGBoost grows trees **level by level**. At depth 1 it splits every node. At depth 2 it splits every node again. Every node at the same depth gets a split attempt regardless of whether that split is actually useful.

The problem: at depth 3, a node that captured the dominant signal in the data might benefit enormously from another split. But a sibling node that captures noise might benefit almost nothing. XGBoost gives them equal priority — it splits both before either gets a second split.

LightGBM grows trees **leaf by leaf**. It maintains a priority queue of all current leaves ranked by their best possible gain. At each step it pops the leaf with the highest gain, splits it, and pushes the two children back. The result: the tree always chases the single most informative split available, regardless of where in the tree it is.

```
Level-wise (XGBoost)         Leaf-wise (LightGBM)

depth 1:   split A           step 1: split A (gain=120)
depth 1:   split B           step 2: split A.left (gain=95)  ← chases the best
depth 2:   split A.left      step 3: split A.left.left (gain=80)
depth 2:   split A.right     step 4: split B (gain=40)
depth 2:   split B.left      ...
depth 2:   split B.right
```

The LightGBM tree is **asymmetric by design**. One branch can be 6 levels deep while another stays at depth 1. It spent its budget where the data warranted it.

## `max_leaves` vs `max_depth`

XGBoost's primary control is `max_depth` — it caps how deep the tree can go. A tree with `max_depth=3` has at most $2^3 = 8$ leaves.

LightGBM's primary control is `max_leaves` — it caps the total number of leaves directly. `max_depth` is still there as a guardrail against extreme asymmetry, but `max_leaves` drives the shape.

This matters because with `max_depth` you can waste capacity. A tree with `max_depth=6` always has room for 64 leaves even if only 10 of them would have positive gain. With `max_leaves=31` (LightGBM's default) you spend exactly 31 leaves — each one earned by having the highest gain among all candidates at that moment.

The default of 31 is not arbitrary — it corresponds roughly to a balanced tree of depth 5 ($2^5 = 32$), but LightGBM can distribute those 31 leaves unevenly across branches.

## The Regressor

The boosting loop is identical to XGBoost.  inherits from  and only overrides  to instantiate  instead, and adds  as the primary tree shape control.  is inherited unchanged.

In [7]:
# dataset where one feature dominates — ideal for leaf-wise growth
np.random.seed(42)
n_samples = 200
X = pd.DataFrame({
    'feature_1': np.random.randn(n_samples),
    'feature_2': np.random.randn(n_samples),
    'feature_3': np.random.randn(n_samples),
    'feature_4': np.random.randn(n_samples),
})

# feature_1 dominates heavily
true_weights = np.array([10.0, 0.5, 0.3, 0.2])
y = X.values @ true_weights + np.random.randn(n_samples) * 0.5

split = int(0.8 * n_samples)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y[:split], y[split:]

## Inspecting Tree Asymmetry

We can traverse the built tree and measure the depth of each leaf to confirm that LightGBM produces asymmetric trees — some branches go deep, others stop early.

In [8]:
def get_leaf_depths(node: Node, depth: int = 0) -> list[int]:
    """Recursively collect the depth of every leaf in the tree."""
    if node.is_leaf:
        return [depth]
    return get_leaf_depths(node.left, depth + 1) + get_leaf_depths(node.right, depth + 1)


def count_leaves(node: Node) -> int:
    if node.is_leaf:
        return 1
    return count_leaves(node.left) + count_leaves(node.right)

## Comparison: XGBoost vs LightGBM

Same dataset, same number of estimators, similar capacity. We compare convergence speed and final accuracy.

In [9]:
lgbm = LightGBMRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_leaves=8,
    max_depth=6,
    min_samples=2,
    reg_lambda=1.0,
    gamma=0.0
)
lgbm.fit(X_train, y_train)
lgbm_pred = lgbm.predict(X_test)

# XGBoost with equivalent capacity: max_depth=3 gives at most 8 leaves
xgb = XGBoostRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=2,
    reg_lambda=1.0,
    gamma=0.0
)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

In [10]:
# inspect asymmetry of the first LightGBM tree
leaf_depths = get_leaf_depths(lgbm.trees[0].tree)
n_leaves = count_leaves(lgbm.trees[0].tree)

print(f"LightGBM tree 1 — leaves: {n_leaves}, depths: {sorted(leaf_depths)}")
print(f"Asymmetry (max - min depth): {max(leaf_depths) - min(leaf_depths)}")
print()

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

print("LightGBM (leaf-wise, max_leaves=8):")
print(f"  RMSE: {rmse(y_test, lgbm_pred):.4f}")
print(f"  R²:   {r2(y_test, lgbm_pred):.4f}")
print()
print("XGBoost  (level-wise, max_depth=3 → ≤8 leaves):")
print(f"  RMSE: {rmse(y_test, xgb_pred):.4f}")
print(f"  R²:   {r2(y_test, xgb_pred):.4f}")

LightGBM tree 1 — leaves: 8, depths: [2, 3, 3, 3, 3, 3, 4, 4]
Asymmetry (max - min depth): 2

LightGBM (leaf-wise, max_leaves=8):
  RMSE: 1.7944
  R²:   0.9613

XGBoost  (level-wise, max_depth=3 → ≤8 leaves):
  RMSE: 1.1370
  R²:   0.9845


## When Does Leaf-wise Win?

Leaf-wise growth has a clear advantage when **a few features dominate**. In those cases the highest-gain splits keep appearing on the same branch — the one tracking the dominant feature. LightGBM follows that branch deep while XGBoost wastes depth on the other branches.

The dataset above was constructed with exactly this property: `feature_1` has weight 10 while the others have weights below 0.5. A well-designed leaf-wise tree will push most of its leaves onto the `feature_1` branch.

When features contribute more **evenly**, the advantage shrinks. Level-wise growth is more systematic in that case — it ensures every region of the feature space gets attention at each depth.

Leaf-wise also carries a **higher overfitting risk** on small datasets. Chasing the best gain greedily can overfit to noise, especially when `max_leaves` is large relative to the number of samples. `min_samples` and `reg_lambda` are the main guards against this.